# AF647 blink photophysics: deconstructing PeakLoc's timing proxy

This notebook asks why `20_temporal_blink_timing_distributions.png` reports ROI event spans that are much longer than an expected Alexa Fluor 647 response. It does not assume that the QC proxy is a molecular lifetime. Instead, it reconstructs the full PeakLoc path for a reproducible random sample of five event-rich, accepted fits per recording:

1. read the retained peak and fitted ROI;
2. seek into the original RAW stream and recover every relevant event;
3. replay the cumulative-polarity interpolation, prominence detection, and spline timing window;
4. reconstruct the exact ROI count bounds from the persisted polarity exposures;
5. verify the raw first/last timestamps and positive/negative counts against the saved ROI;
6. isolate dense, spatially compact ON/OFF event trains around the retained peak; and
7. export publication-ready figures and machine-readable train/interval source data.

The coordinate convention is `image[y, x]`; all image overlays use `scatter(sub_x, sub_y)`.

In [ ]:
from __future__ import annotations

import csv
from pathlib import Path
import sys

from IPython.display import Image, Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    path = (Path.cwd() if start is None else start).resolve()
    for candidate in (path, *path.parents):
        if (candidate / "pixi.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the PeakLoc repository root")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from localization_scripts.photophysics_deconstruction import (  # noqa: E402
    analyze_run,
    discover_completed_runs,
)

REPO_ROOT

## What an AF647 comparison can and cannot say

AF647 ON-state kinetics are condition-dependent. Lin *et al.* measured exponential ON-time distributions and reported lifetimes of only a few milliseconds at their high excitation intensities, while emphasizing dependence on excitation and local chemistry ([PLOS ONE 2015](https://doi.org/10.1371/journal.pone.0128135)). Diekmann *et al.* likewise showed that excitation intensity changes AF647 blinking and localization performance ([Nature Methods 2020](https://doi.org/10.1038/s41592-020-0918-5)).

An event camera reports asynchronous log-intensity threshold crossings, not direct fluorescence-state occupancy ([Gallego *et al.*, IEEE TPAMI 2022](https://doi.org/10.1109/TPAMI.2020.3008413)). Therefore, a long ROI first-to-last event span is evidence about the detector/algorithm window, background, or overlapping emitters—not by itself evidence for a long molecular ON state. A molecular lifetime comparison would additionally require matched excitation intensity, buffer composition, camera bias/threshold calibration, and a generative kinetic model.

In [ ]:
INPUT_PATHS = [
    REPO_ROOT
    / "data/2026_07_15_Microtubule_Recordings/Normal_vs_Rapid/Rapid_Blinking_Only"
    / "recording_2026-07-15_14-01-32/20260716_193311_378386",
    REPO_ROOT
    / "data/2026_07_15_Microtubule_Recordings/Normal_vs_Rapid"
    / "405_Induced_Rapid_Switching",
    REPO_ROOT / "data/2025_rec",
]
SAMPLE_SIZE = 5
RANDOM_SEED = 647

run_directories = discover_completed_runs(INPUT_PATHS)
print(f"Resolved {len(run_directories)} completed runs:")
for run_directory in run_directories:
    print(f"- {run_directory.relative_to(REPO_ROOT)}")

## Sampling and exact timing definitions

The sampling population is the accepted-localization array, so every sampled blink has already passed the joint Poisson fit and QC filters. “Good amount of events” is made reproducible: retain the 60th–90th percentile of total fit-event counts, then sample uniformly without replacement using seed 647. The upper bound avoids selecting only pathological event-count extremes.

For each selected ROI, PeakLoc's persisted quantities mean:

- **turn-on proxy:** absolute `t_1st`, the earliest event anywhere in the spatial ROI and count window; the relative rise proxy is `t_peak - t_1st`;
- **on-duration proxy:** `t_last - t_1st`;
- **turn-off proxy:** absolute `t_last`, the latest event anywhere in the spatial ROI and count window; the relative decay proxy is `t_last - t_peak`.

The ±5 ms polarity gate is not a symmetric crop. Positive events satisfy `t < peak + gate`; negative events satisfy `t > peak - gate`. Those one-sided tests classify the two polarity lobes while the spline-derived count window can remain hundreds of milliseconds or longer.

The segmented estimator instead reports the first event in the dense positive ON train, the last event in its spatially matched negative OFF train, and their full cycle span. It rejects weak, diffuse, missing-polarity, and ambiguous candidates. Segmented event maps are compared with legacy fit events using explicit overlap, newly included, and excluded counts, but are not refitted in this notebook.

Only the exact per-sample legacy window and segmentation context are read from RAW. Legacy counts/timestamps remain exactly reproducible; the displayed spline trace is a local-context diagnostic, not a bitwise replay of the original full slice.

In [ ]:
artifacts = []
for run_directory in run_directories:
    print(f"Analyzing {run_directory.relative_to(REPO_ROOT)} ...")
    result = analyze_run(
        run_directory,
        sample_size=SAMPLE_SIZE,
        random_seed=RANDOM_SEED,
    )
    artifacts.append(result)
    print(
        f"  median ROI proxy = {result.median_roi_duration_ms:.1f} ms; "
        f"segmented = {result.segmented_sample_count}/{result.sample_count}, median {result.median_segmented_cycle_ms:.1f} ms; "
        f"RAW validation = {result.all_validations_passed}"
    )
    print(f"  wrote {result.output_directory.relative_to(REPO_ROOT)}")

In [ ]:
summary_lines = [
    "| Recording | Blinks | Legacy median (ms) | Segmented accepted | Segmented median (ms) | Exact RAW checks |",
    "|---|---:|---:|---:|---:|:---:|",
]
for artifact in artifacts:
    summary_lines.append(
        "| "
        + artifact.run_directory.parent.name
        + f" | {artifact.sample_count} | {artifact.median_roi_duration_ms:.1f} "
        + f"| {artifact.segmented_sample_count} | {artifact.median_segmented_cycle_ms:.1f} "
        + f"| {'pass' if artifact.all_validations_passed else 'FAIL'} |"
    )
display(Markdown("\n".join(summary_lines)))

## How to read the outputs

Each blink deconstruction has five panels: positive and negative legacy event-count PSFs; the local-context detection trace; a ±250 ms raster with selected ON/OFF supports and core-train events; and a timing schematic comparing legacy and segmented spans. The 3D figure exposes spatially dispersed tail events.

Plotted values are in `timing_summary.csv`, `raw_roi_events.csv`, `segmentation_context_events.csv`, `detection_trace.csv`, `transition_trains.csv`, `blink_intervals.csv`, `temporal_activity_bins.csv`, `segmented_fit_events.csv`, and `nearby_seed_intervals.csv`. The nearby-seed figure evaluates each retained peak independently; cycle ownership is not yet deduplicated. PDFs preserve vector text and geometry; PNGs are publication resolution.

In [ ]:
for artifact in artifacts:
    display(Markdown(f"### `{artifact.run_directory.parent.name}`"))
    display(Image(filename=artifact.output_directory / "sample_overview.png"))
    display(Image(filename=artifact.output_directory / "blink_01_deconstruction.png"))
    display(Image(filename=artifact.output_directory / "nearby_seed_intervals.png"))
    display(Image(filename=artifact.output_directory / "roi_event_clouds_3d.png"))

In [ ]:
first_event_table = artifacts[0].output_directory / "raw_roi_events.csv"
with first_event_table.open(encoding="utf-8") as input_file:
    first_rows = list(csv.DictReader(input_file))[:12]

timestamp_lines = [
    "| sample | event | t (us) | t - peak (us) | x | y | polarity |",
    "|---:|---:|---:|---:|---:|---:|---:|",
]
for row in first_rows:
    timestamp_lines.append(
        f"| {row['sample_id']} | {row['event_index']} | {row['t_us']} | "
        f"{row['t_relative_to_peak_us']} | {row['x_px']} | {row['y_px']} | "
        f"{row['polarity']} |"
    )
display(
    Markdown(
        "### First raw timestamps (complete table is in the QC folder)\n\n"
        + "\n".join(timestamp_lines)
    )
)

In [ ]:
for artifact in artifacts:
    report = (artifact.output_directory / "README.md").read_text(encoding="utf-8")
    display(Markdown(report))

## Interpretation

The decisive test is the exact RAW reconstruction: if `t_1st`, `t_last`, and both polarity counts reproduce the stored ROI, the long timing value is not a plotting-unit error. It is the expected result of applying a first-to-last statistic to all retained events across a broad, spline-derived ROI window. Sparse background events, nearby emitters within the 13 × 13 fit ROI, and the detector's contrast-threshold dynamics can all lengthen that span.

The dense-train segmenter provides a stricter algorithmic estimate: it separates a compact positive onset train from its spatially matched negative offset train and rejects candidates without both. These cycle spans remain event-camera transition proxies—not direct AF647 state lifetimes—and the selected maps require a separate refit/QC pass before localization-accuracy claims.